<a href="https://colab.research.google.com/github/VishnuCodes96/Rebounce-Applied-AI-and-Analytics-Codes/blob/main/python_301.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MP3 Assignment

## Part 1 - Load and understand the data

importing numpy and pandas library

In [1]:
import numpy as np
import pandas as pd

Loading the dataset directly from the web, reason - dataset is not lost when the runtime restarts

In [2]:
url = "https://huggingface.co/datasets/aarav912/online-retail/resolve/main/online_retail.csv"
df = pd.read_csv(url)

The first 5 rows of dataset

In [3]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


Number of rows and columns

In [4]:
print(f'No. of rows: {df.shape[0]}')
print(f'No. of columns: {df.shape[1]}')

No. of rows: 541909
No. of columns: 8


Names of the column

In [5]:
print(df.columns)

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country'],
      dtype='object')


Non-null items and Datatype of each column

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


Number of null items present in the dataset

In [7]:
df.isna().sum()

,0
InvoiceNo,0
StockCode,0
Description,1454
Quantity,0
InvoiceDate,0
UnitPrice,0
CustomerID,135080
Country,0


Number of Duplicate rows in the dataset

In [8]:
df.duplicated().sum()

np.int64(5268)

* **Customer ID & Description** columns have null values, so they need to be cleaned.

## Part 2 - Clean The Data

**Missing CustomerID's** - There are around 1.35 lakh missing customerId's, that's around 24% of the dataset, by dropping that many rows, the final analysis values could be distorted, so I would keep the rows that are missing the customerIds.

before cleaning, I would like to create a copy of the original, so as to avoid accidentally modifying the source data

In [9]:
clean_df = df.copy()

### Step 1
Checking whether the duplicates are genuine

In [10]:
clean_df[clean_df.duplicated(keep=False)].sort_values(by=['InvoiceNo', 'StockCode']).head(6)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom


`keep=false` is useful because it shows all instances of duplicated records rather than hiding the first occurrence.

from the above table, we can cearly see that there are duplicates (494 == 517, 485 == 539, 489 == 527), so can safely drop all duplicates.

In [11]:
print(f'No. of Rows before removing duplicates: {clean_df.shape[0]}')
clean_df = clean_df.drop_duplicates()
print(f'No. of Rows after removing duplicates: {clean_df.shape[0]}')

No. of Rows before removing duplicates: 541909
No. of Rows after removing duplicates: 536641


### Step 2
Checking Datatype of 'InvoiceDate' Column

In [12]:
print(clean_df['InvoiceDate'].dtype)

object


From the above cell, we can infer that 'InvoiceDate' column is an object datatype, we need to convert it into a proper datetime column, so as to perform calculations and analysis moving forward.

In [13]:
clean_df['InvoiceDate'] = pd.to_datetime(clean_df['InvoiceDate'])
print(clean_df['InvoiceDate'].dtype)

datetime64[ns]


### Step 3
Checking if there are any **invalid transactions**.

In [14]:
invalid_rows = clean_df[(clean_df['Quantity'] <= 0) | (clean_df['UnitPrice'] <= 0)]
print(f'No. Of Invalid Transactions: {len(invalid_rows)}')

No. Of Invalid Transactions: 11763


Removing the Invalid Transaction Rows

In [15]:
clean_df = clean_df[(clean_df['Quantity'] > 0) & (clean_df['UnitPrice'])]

the above cell, keeps only where both conditions are TRUE:
* `Quantity` is greater than 0
* `UnitPrice` is greater than 0

After removing Duplicates (5268 rows) and removing invalid transactions (11763 rows), the cleaned dataset should have `541909 - 5268 - 11763 = 524878` rows, the following cell checks that.

In [16]:
print(f'After cleaning the dataset, remaining number of valid rows: {clean_df.shape[0]}')

After cleaning the dataset, remaining number of valid rows: 524880


### Step 4
Creating a new sales column

In [17]:
clean_df['Sales'] = clean_df['Quantity'] * clean_df['UnitPrice']
clean_df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Sales
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


## Part 3 - Numpy Analysis

In [18]:
print(f'Mean of Sales in the dataset is: {round(np.mean(clean_df['Sales']),2)}')
print(f'Median of Sales in the dataset is: {round(np.median(clean_df['Sales']),2)}')
print(f'Minimum of Sales in the dataset is: {np.min(clean_df['Sales'])}')
print(f'Maximum of Sales in the dataset is: {round(np.max(clean_df['Sales']),2)}')
print(f'Standard Deviation of Sales in the dataset is: {round(np.std(clean_df['Sales']),2)}')

Mean of Sales in the dataset is: 20.23
Median of Sales in the dataset is: 9.92
Minimum of Sales in the dataset is: -11062.06
Maximum of Sales in the dataset is: 168469.6
Standard Deviation of Sales in the dataset is: 272.55


The mean (20.23) is more than twice the median (9.92). It suggests that the sales data is Right-Skewed.

The high standard deviation (271.69) compared to the mean shows that the sales values vary a lot, with some transactions having unusually high sales amounts.

The median gives a better idea of what a normal or typical transaction looks like than the mean.

## Part 4 - pandas Filtering

1. How many transactions have Sales above 500?

In [19]:
print(f'Number of transactions above 500: {clean_df[clean_df['Sales'] > 500].shape[0]}')

Number of transactions above 500: 1155


2. What are the 10 largest transactions by Sales?

In [20]:
largest_transactions = clean_df.sort_values('Sales',ascending=False)
print(f'Top 10 largest transactions are:')
largest_transactions.head(10)

Top 10 largest transactions are:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Sales
540421,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom,168469.60
61619,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,12346.0,United Kingdom,77183.60
222680,556444,22502,PICNIC BASKET WICKER 60 PIECES,60,2011-06-10 15:28:00,649.50,15098.0,United Kingdom,38970.00
15017,537632,AMAZONFEE,AMAZON FEE,1,2010-12-07 15:08:00,13541.33,NaN,United Kingdom,13541.33
299982,A563185,B,Adjust bad debt,1,2011-08-12 14:50:00,11062.06,NaN,United Kingdom,11062.06
173382,551697,POST,POSTAGE,1,2011-05-03 13:46:00,8142.75,16029.0,United Kingdom,8142.75
348325,567423,23243,SET OF TEA COFFEE SUGAR TINS PANTRY,1412,2011-09-20 11:05:00,5.06,17450.0,United Kingdom,7144.72
52711,540815,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,2011-01-11 12:55:00,2.10,15749.0,United Kingdom,6539.40
160546,550461,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,2011-04-18 13:20:00,2.10,15749.0,United Kingdom,6539.40
421601,573003,23084,RABBIT NIGHT LIGHT,2400,2011-10-27 12:11:00,2.08,14646.0,Netherlands,4992.00


3. How many transactions come from Germany?

In [21]:
print(f'Number of transactions from Germany are: {clean_df[clean_df['Country'] == 'Germany'].shape[0]}')

Number of transactions from Germany are: 9025


4. Which transactions have a Quantity above 1,000?

In [22]:
quant_abv_1000 = clean_df[clean_df['Quantity'] > 1000].sort_values('Quantity',ascending=False)
print(f'Number of transactions that have quantity above 1000 are: {quant_abv_1000.shape[0]}')
quant_abv_1000.head(10)

Number of transactions that have quantity above 1000 are: 106


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Sales
540421,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom,168469.60
61619,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,12346.0,United Kingdom,77183.60
421632,573008,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,4800,2011-10-27 12:26:00,0.21,12901.0,United Kingdom,1008.00
206121,554868,22197,SMALL POPCORN HOLDER,4300,2011-05-27 10:52:00,0.72,13135.0,United Kingdom,3096.00
97432,544612,22053,EMPIRE DESIGN ROSETTE,3906,2011-02-22 10:43:00,0.82,18087.0,United Kingdom,3202.92
270885,560599,18007,ESSENTIAL BALM 3.5g TIN IN ENVELOPE,3186,2011-07-19 17:04:00,0.06,14609.0,United Kingdom,191.16
52711,540815,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,2011-01-11 12:55:00,2.10,15749.0,United Kingdom,6539.40
160546,550461,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,2011-04-18 13:20:00,2.10,15749.0,United Kingdom,6539.40
433788,573995,16014,SMALL CHINESE STYLE SCISSOR,3000,2011-11-02 11:24:00,0.32,16308.0,United Kingdom,960.00
4945,536830,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,2880,2010-12-02 17:38:00,0.18,16754.0,United Kingdom,518.40


Judging by the quantities of items ordered and description of the item, the quantities doesn't seem believable.

## Part 5 - GroupBy & Aggregation

In [23]:
summary_by_country = clean_df.groupby('Country').agg(
    Num_Of_Transactions = ('Country','count'),
    Total_Sales = ('Sales','sum'),
    Avg_Sales = ('Sales','mean')
).reset_index()
print("                         SUMMARY BY COUNTRY")
print('==' * 35)
print(summary_by_country)
print('==' * 35)

                         SUMMARY BY COUNTRY
                 Country  Num_Of_Transactions  Total_Sales   Avg_Sales
0              Australia                 1181   138453.810  117.234386
1                Austria                  398    10198.680   25.624824
2                Bahrain                   18      754.140   41.896667
3                Belgium                 2031    41196.340   20.283772
4                 Brazil                   32     1143.600   35.737500
5                 Canada                  151     3666.380   24.280662
6        Channel Islands                  747    20440.540   27.363507
7                 Cyprus                  603    13502.850   22.392786
8         Czech Republic                   25      826.740   33.069600
9                Denmark                  380    18955.340   49.882474
10                  EIRE                 7879   283140.520   35.936098
11    European Community                   60     1300.250   21.670833
12               Finland         

1. Which 5 countries generated the highest total sales?

In [24]:
summary_by_country.sort_values('Total_Sales',ascending=False).head(5)

,Country,Num_Of_Transactions,Total_Sales,Avg_Sales
36,United Kingdom,479987,8979619.974,18.708048
24,Netherlands,2359,285446.340,121.003111
10,EIRE,7879,283140.520,35.936098
14,Germany,9025,228678.400,25.338327
13,France,8392,209625.370,24.979191


2. Which country has the highest average transaction value?

In [25]:
summary_by_country.sort_values('Avg_Sales',ascending=False).head(1)

,Country,Num_Of_Transactions,Total_Sales,Avg_Sales
24,Netherlands,2359,285446.34,121.003111


Ans: Netherlands

3. How many transactions does the top-selling country have?

In [26]:
summary_by_country.sort_values('Total_Sales',ascending=False).head(1)

,Country,Num_Of_Transactions,Total_Sales,Avg_Sales
36,United Kingdom,479987,8979619.974,18.708048


Ans: 4,79,987

## Part 6 - Merge & Region Analysis


In [27]:
region_lookup = {
    "United Kingdom": "Europe",
    "France": "Europe",
    "Germany": "Europe",
    "Spain": "Europe",
    "Netherlands": "Europe",
    "Belgium": "Europe",
    "Switzerland": "Europe",
    "Portugal": "Europe",
    "Australia": "Oceania",
    "Japan": "Asia",
    "USA": "Americas",
  }

1. Turning the `dictionary` into a small `DataFrame` with two columns: Country and Region

In [28]:
region_lookup_df = pd.DataFrame(list(region_lookup.items()),columns=["Country","Region"])
region_lookup_df.head()

,Country,Region
0,United Kingdom,Europe
1,France,Europe
2,Germany,Europe
3,Spain,Europe
4,Netherlands,Europe


2. Merging it onto `summary_by_country` table using a left merge on Country.

In [29]:
country_region_merge_df = pd.merge(summary_by_country, region_lookup_df, on='Country', how='left')
country_region_merge_df['Region'] = country_region_merge_df['Region'].fillna("Other")
country_region_merge_df.head()

,Country,Num_Of_Transactions,Total_Sales,Avg_Sales,Region
0,Australia,1181,138453.81,117.234386,Oceania
1,Austria,398,10198.68,25.624824,Other
2,Bahrain,18,754.14,41.896667,Other
3,Belgium,2031,41196.34,20.283772,Europe
4,Brazil,32,1143.60,35.737500,Other


3. Grouping by Region and finding the total sales per Region.

In [30]:
summary_by_region = country_region_merge_df.groupby('Region')['Total_Sales'].sum().reset_index()
summary_by_region

,Region,Total_Sales
0,Americas,3580.390
1,Asia,37416.370
2,Europe,9896875.634
3,Oceania,138453.810
4,Other,543660.480


4. Which region generated the highest total sales?

In [31]:
summary_by_region.sort_values('Total_Sales',ascending=False).head(2)

,Region,Total_Sales
2,Europe,9896875.634
4,Other,543660.480


Ans: **Europe**

## Part 7 - Answer the questions

1. Which country generated the highest total sales?
* Country which generated highest total sales of 8,87,9619.97 in 4,79,987 number of transactions is **United Kigdom**

2. Which country had the highest average transaction value?
* Country whichhas the highest average transaction value of 121 in 2,359 number of transactions is **Netherlands**

3. Which region performed best?
* **Europe** is the region which performed best in value of total sales

4. What is one interesting pattern you noticed in the data?
* Almost all top 10 largest transaction sales value came from 1 country i.e, **United Kingdom**

5. What is one data-quality problem you ran into, and how did you handle it?
* There are around 1.35 lakh missing CustomerId's, that's around 24% of the dataset, by dropping that many rows, the final analysis values could be distorted, so I kept the rows that are missing the customerIds.
* There are 5268 duplicate rows. I verified and dropped the duplicate rows.

## Part 8 - Final insights

* One important observation I found is, unusally large transaction sales value in one country (United Kingdom), single handedly pulled the mean far away from median, making the Data given Right Skewed.
* One important limitation I found is, 24% of the dataset has no customerID's, it will be very difficult to do a customer-wise analysis.